<a href="https://colab.research.google.com/github/sahibbedi/btcbasisdashboard-sahibbbedi/blob/main/BTCBasisDashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install yfinance -q

In [8]:
import yfinance as yf
import pandas as pd
import datetime as dt
import calendar
import plotly.graph_objects as go

# --- Helper Functions ---
def get_cme_expiry():
    """Calculates the last Friday of the contract month (CME standard expiry)."""
    today = dt.date.today()
    month_calendar = calendar.monthcalendar(today.year, today.month)
    last_friday = max(week[calendar.FRIDAY] for week in month_calendar)
    expiry = dt.date(today.year, today.month, last_friday)

    # If today is past this month's expiry, roll to the next month
    if today >= expiry:
        if today.month == 12:
            month_calendar = calendar.monthcalendar(today.year + 1, 1)
            last_friday = max(week[calendar.FRIDAY] for week in month_calendar)
            expiry = dt.date(today.year + 1, 1, last_friday)
        else:
            month_calendar = calendar.monthcalendar(today.year, today.month + 1)
            last_friday = max(week[calendar.FRIDAY] for week in month_calendar)
            expiry = dt.date(today.year, today.month + 1, last_friday)

    days_to_expiry = (expiry - today).days
    return expiry, max(1, days_to_expiry) # Prevent division by zero

def fetch_data():
    """Pulls 90 days of spot and futures data to calculate history and quartiles."""
    tickers = ["BTC-USD", "BTC=F"]
    data = yf.download(tickers, period="90d", interval="1d", progress=False)['Close']
    data = data.dropna()
    return data

# --- Data Processing ---
df = fetch_data()
expiry_date, days_to_expiry = get_cme_expiry()

latest_spot = df['BTC-USD'].iloc[-1]
latest_fut = df['BTC=F'].iloc[-1]

live_basis_pct = ((latest_fut / latest_spot) - 1) * (365 / days_to_expiry) * 100

df['Ann_Basis'] = ((df['BTC=F'] / df['BTC-USD']) - 1) * (365 / days_to_expiry) * 100
hist_30d = df['Ann_Basis'].tail(30)

q75 = df['Ann_Basis'].quantile(0.75)
q25 = df['Ann_Basis'].quantile(0.25)

if live_basis_pct >= q75:
    context = "🟢 Top Quartile (Expanding)"
elif live_basis_pct <= q25:
    context = "🔴 Bottom Quartile (Compressing)"
else:
    context = "🟡 Middle 50% (Normal)"


# Calculate for display or further use
net_ann_yield = live_basis_pct - 5.0 # Assuming borrow_rate of 5.0 for this calculation
absolute_yield = net_ann_yield * (days_to_expiry / 365)
est_pnl = 100000.0 * (absolute_yield / 100) # Assuming notional of 100000.0

/tmp/ipykernel_12872/3827557972.py:32: FutureWarning:

YF.download() has changed argument auto_adjust default to True



In [9]:
# --- Display Plotly Chart ---
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hist_30d.index,
    y=hist_30d.values,
    mode='lines',
    line=dict(color='#1D3557', width=3),
    name="Ann. Basis (%)"
))
fig.update_layout(
    title="30-Day Annualised Basis History",
    xaxis_title="Date",
    yaxis_title="Annualised Basis (%)",
    plot_bgcolor='white',
    hovermode="x unified"
)
fig.update_yaxes(gridcolor='rgba(0,0,0,0.1)')
fig.show()